# 04. University of Arizona 4 km Operational Partial-Monthly SWE Pipeline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/flowracle/notebooks/data_preprocessing/04_download_ua_swe_monthly.ipynb)

This notebook downloads daily 4 km Snow Water Equivalent (SWE) and snow depth
grids from the University of Arizona (`https://climate.arizona.edu/data/UA_SWE/DailyData_4km`),
computes **causal partial-monthly averages over days $1 \dots (N - 1)$** of
each calendar month, crops to the extended California snow-fed domain
(`32.5–45.5°N`, `124.0–115.0°W`), and saves to Zarr:
- **Output path**:
  `${FLOWRACLE_DATA_DIR}/ua-swe-ca-region-operational-monthly-averages/ua_swe_CA_region.zarr`

### Operational Causality Note
Omitting the final calendar day $N$ of month $m-1$ ensures the monthly SWE
summary is strictly available prior to forecast issuance on day 1 of month $m$,
respecting operational assimilation latency.

##### Copyright 2026 The Google Research Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

---
*Disclaimer: This is not an officially supported Google product.*

In [ ]:
!pip install -q "numpy>=1.26.0" "pandas>=2.0.0" "requests>=2.31.0" "beautifulsoup4>=4.12.0" "lxml>=4.9.0" "xarray>=2023.12.0" "zarr>=2.16.0,<3.0.0" "dask>=2023.12.0" "tqdm>=4.66.0" "netCDF4>=1.6.0"

In [ ]:
import datetime
import os
import tempfile

import netCDF4  # noqa: F401
import numpy as np
import pandas as pd
import requests
from tqdm import tqdm
import xarray as xr

FLOWRACLE_DATA_DIR = os.environ.get("FLOWRACLE_DATA_DIR", "./flowracle_data")
OUTPUT_DIR = os.path.join(
    FLOWRACLE_DATA_DIR, "ua-swe-ca-region-operational-monthly-averages"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)
ZARR_STORE_PATH = os.path.join(OUTPUT_DIR, "ua_swe_CA_region.zarr")

BASE_HTTP_URL = "https://climate.arizona.edu/data/UA_SWE/DailyData_4km"
LAT_MIN, LAT_MAX = 32.5, 45.5
LON_MIN, LON_MAX = -124.0, -115.0
START_DATE = "1981-10-01"
END_DATE = (datetime.datetime.now() - pd.DateOffset(months=1)).strftime(
    "%Y-%m-%d"
)

## 1. Daily Download & Partial-Monthly Aggregation Functions

In [ ]:
def download_daily_ua_file(year, month, day, temp_dir):
  wy = year + 1 if month >= 10 else year
  date_str = f"{year}{month:02d}{day:02d}"
  local_path = os.path.join(temp_dir, f"UA_SWE_{date_str}.nc")
  for suffix in ("stable", "provisional", "early"):
    url = f"{BASE_HTTP_URL}/WY{wy}/UA_SWE_Depth_4km_v1_{date_str}_{suffix}.nc"
    try:
      resp = requests.get(url, stream=True, timeout=60)
      if resp.status_code == 200:
        with open(local_path, "wb") as f:
          for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
        return local_path
    except requests.RequestException:
      continue
  return None


def process_partial_month_ca(year, month, zarr_path):
  month_start = pd.Timestamp(f"{year}-{month:02d}-01")
  if os.path.exists(zarr_path):
    ds_existing = xr.open_zarr(zarr_path)
    already_done = month_start in pd.to_datetime(ds_existing.time.values)
    ds_existing.close()
    if already_done:
      return None

  num_days = pd.Period(f"{year}-{month:02d}").days_in_month
  target_days = range(1, num_days)  # Days 1 .. N-1 for operational causality
  sum_swe, count = None, 0

  with tempfile.TemporaryDirectory() as temp_dir:
    for day in tqdm(target_days, desc=f"{year}-{month:02d}", leave=False):
      nc_path = download_daily_ua_file(year, month, day, temp_dir)
      if not nc_path:
        continue
      try:
        ds = xr.open_dataset(nc_path).load()
        num_vars = [
            v for v in ds.data_vars if np.issubdtype(ds[v].dtype, np.number)
        ]
        cropped = (
            ds[num_vars]
            .squeeze("time", drop=True)
            .sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
        )
        sum_swe = (
            cropped.copy(deep=True) if sum_swe is None else (sum_swe + cropped)
        )
        count += 1
        ds.close()
      finally:
        if os.path.exists(nc_path):
          os.remove(nc_path)

  if count == 0:
    return None
  mean_ds = (sum_swe / count).expand_dims(time=[month_start])
  mean_ds.attrs = {
      "source": (
          "University of Arizona (UA) 4km Snow Water Equivalent (SWE) Daily"
          " Data"
      ),
      "url": BASE_HTTP_URL,
      "description": (
          "Partial monthly average of SWE depth over days 1 to (N-1), omitting"
          " the last day of each month for 1st-of-month operational causality."
      ),
  }
  return mean_ds


def append_months_to_zarr(ds_list, zarr_path):
  if not ds_list:
    return
  combined = xr.concat(ds_list, dim="time")
  if not os.path.exists(zarr_path):
    enc = {
        v: {"chunks": (12,) + combined[v].shape[1:]} for v in combined.data_vars
    }
    combined.to_zarr(zarr_path, mode="w", encoding=enc)
  else:
    combined.to_zarr(zarr_path, mode="a", append_dim="time")

## 2. Execute Resumable Download Loop (Oct 1981 – Present)

In [ ]:
months = pd.date_range(start=START_DATE, end=END_DATE, freq="MS")
buffer = []
for ts in months:
  ds_m = process_partial_month_ca(ts.year, ts.month, ZARR_STORE_PATH)
  if ds_m is not None:
    buffer.append(ds_m)
  if ts.month == 12 and buffer:
    append_months_to_zarr(buffer, ZARR_STORE_PATH)
    print(f"Saved through {ts.year}-12 ({len(buffer)} new months).")
    buffer = []

if buffer:
  append_months_to_zarr(buffer, ZARR_STORE_PATH)

print(xr.open_zarr(ZARR_STORE_PATH))